In [3]:
print("Pene de Lur")
print("Nigga")
print("Hola Luristán")
print("iñigo therian")

Pene de Lur
Nigga
Hola Luristán
iñigo therian


In [2]:
### IMPORTS ###
import pandas as pd
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
import optuna
from optuna.pruners import MedianPruner
import xgboost as xgb
from sklearn.ensemble import (
    VotingClassifier, BaggingClassifier, AdaBoostClassifier,
    GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report,
    precision_score, recall_score, f1_score, matthews_corrcoef
)
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

DATA PROCESSING

In [3]:
### DATASET LOADING ###

#Lo antiguo de Jon
#gauth = GoogleAuth()
#gauth.LocalWebserverAuth()
#drive = GoogleDrive(gauth)

# Google Drive ID for public sharing of the dataset
#file_id = "1aGGJNl1VtET_6HqQh81ouSKqe6-lN-OG"
#file = drive.CreateFile({"id": file_id})
#file.GetContentFile("ai_tools.csv")

# Reading the csv and loading it into a pandas dataframe.
#df = pd.read_csv("ai_tools.csv")
#df.head()

url = "https://raw.githubusercontent.com/GafillaAbrahamer/ClassificationProject/refs/heads/main/AI_Landscape_19k_Tools_2026.csv"
df = pd.read_csv(url)
df.head()

# The target is already a categorical multiclass column — no transformation needed.
# Classes: "Free", "Freemium", "Subscription", "Pay-as-you-go", "Open Source", "Usage-Based"

# We remove the target from the features to avoid data leakage.
X = df.drop(["Pricing_Model"], axis=1)

# We only keep the label.
y = df[["Pricing_Model"]]

print(X)
print(y)

               AI_Name     Developer  Release_Year  \
0             Scrip Ai       Scripai          2024   
1             Quickads      Quickads          2025   
2           Wonderchat    Wonderchat          2024   
3         Creatosaurus  Creatosaurus          2023   
4                Blobr         Blobr          2025   
...                ...           ...           ...   
19324         Emozi Ai         Emozi          2025   
19325        Myresumai     Myresumai          2022   
19326  Code Screenshot            Cs          2024   
19327      Ho Ho Hello     Hohohello          2022   
19328             Puti          Puti          2024   

                 Intelligence_Type Primary_Domain  \
0                 Generative Video          Video   
1      Computer Vision / Diffusion   Image/Design   
2         Multimodal Generative AI  General/Other   
3                 Autonomous Agent     Automation   
4                 Autonomous Agent     Automation   
...                            ..

In [4]:
### MISSING DATA ANALYSIS ###

# Total missing values per column
print("Missing values per column:")
print(df.isnull().sum())

# Percentage of missing values per column
print("\nMissing values (%):")
print((df.isnull().sum() / len(df) * 100).round(2))

# Quick summary: any missing at all?
print(f"\nTotal missing values in dataset: {df.isnull().sum().sum()}")

Missing values per column:
AI_Name                 0
Developer               0
Release_Year            0
Intelligence_Type       0
Primary_Domain          0
Key_Functionality       0
Pricing_Model           0
API_Availability        0
Context_Window       3877
Accessibility           0
Website_URL             0
Popularity_Votes        0
dtype: int64

Missing values (%):
AI_Name               0.00
Developer             0.00
Release_Year          0.00
Intelligence_Type     0.00
Primary_Domain        0.00
Key_Functionality     0.00
Pricing_Model         0.00
API_Availability      0.00
Context_Window       20.06
Accessibility         0.00
Website_URL           0.00
Popularity_Votes      0.00
dtype: float64

Total missing values in dataset: 3877


In [5]:
### TRANSFORMATION PIPELINE ###
# We drop non-informative columns
cols_to_drop = ['AI_Name', 'Developer', 'Website_URL', 'Key_Functionality']
X = df.drop(columns=cols_to_drop + ['Pricing_Model'])
y = df['Pricing_Model']
X['Context_Window'] = X['Context_Window'].fillna('N/A')

# Train / Validation / Test split (80-10-10)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train size:      {X_train.shape[0]}")
print(f"Validation size: {X_val.shape[0]}")
print(f"Test size:       {X_test.shape[0]}")

# We define column groups for encoding
ordinal_cols = ['Context_Window']
ordinal_categories = [['N/A', '8k', '32k', '128k', '1M']]  # explicit order

onehot_cols = ['Intelligence_Type', 'Primary_Domain', 'API_Availability', 'Accessibility']

numerical_cols = ['Release_Year', 'Popularity_Votes']

# we build the ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('ordinal', OrdinalEncoder(categories=ordinal_categories, handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), onehot_cols),
    ('scaler', StandardScaler(), numerical_cols),
])

# we fit ONLY on train, then transform each split
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
X_test_processed  = preprocessor.transform(X_test)

# Optional: recover a readable DataFrame
onehot_feature_names = preprocessor.named_transformers_['onehot'].get_feature_names_out(onehot_cols)
feature_names = ordinal_cols + list(onehot_feature_names) + numerical_cols

X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names)
X_val_processed   = pd.DataFrame(X_val_processed,   columns=feature_names)
X_test_processed  = pd.DataFrame(X_test_processed,  columns=feature_names)

print(X_train_processed.head())
print(f"\nFinal shape — Train: {X_train_processed.shape}, Val: {X_val_processed.shape}, Test: {X_test_processed.shape}")

Train size:      15463
Validation size: 1933
Test size:       1933
   Context_Window  Intelligence_Type_Autonomous Agent  \
0             3.0                                 0.0   
1             0.0                                 1.0   
2             3.0                                 0.0   
3             3.0                                 0.0   
4             4.0                                 0.0   

   Intelligence_Type_Code Intelligence  \
0                                  0.0   
1                                  0.0   
2                                  0.0   
3                                  0.0   
4                                  0.0   

   Intelligence_Type_Computer Vision / Diffusion  \
0                                            0.0   
1                                            0.0   
2                                            0.0   
3                                            1.0   
4                                            0.0   

   Intelligence_Type_Gen

IMBALANCE ANALYSIS AND MITIGATION

MODEL COMPARISON AND HYPERPARAMETER TUNING

In [ ]:
### POINT 3: MODEL COMPARISON + HYPERPARAMETER TUNING WITH OPTUNA ###

# XGBoost requires numeric class labels — it cannot handle string labels like
# 'Free', 'Freemium', etc. We encode y_train and y_test to integers using
# LabelEncoder. Scikit-learn models work fine with both strings and integers,
# so using y_train_encoded for all models is safe and consistent.
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# ── OPTUNA OBJECTIVE ──────────────────────────────────────────────────────────

def objective(trial):

    classifier_name = trial.suggest_categorical("classifier", [
        "VotingHard", "VotingSoft", "Bagging", "AdaBoost",
        "GradientBoosting", "XGBoost", "RandomForest", "Stacking"
    ])

    if classifier_name == "VotingHard":
        clf = VotingClassifier(
            estimators=[
                ('lr', LogisticRegression(max_iter=1000, random_state=42)),
                ('rf', RandomForestClassifier(
                    n_estimators=trial.suggest_int("voting_hard_rf_n_estimators", 50, 200),
                    random_state=42
                )),
                ('svc', SVC(random_state=42))
            ],
            voting='hard'
        )

    elif classifier_name == "VotingSoft":
        clf = VotingClassifier(
            estimators=[
                ('lr', LogisticRegression(max_iter=1000, random_state=42)),
                ('rf', RandomForestClassifier(
                    n_estimators=trial.suggest_int("voting_soft_rf_n_estimators", 50, 200),
                    random_state=42
                )),
                # probability=True is required for soft voting — SVC does not
                # output probabilities by default, but soft voting needs them
                # to average class probabilities across estimators.
                ('svc', SVC(probability=True, random_state=42))
            ],
            voting='soft'
        )

    elif classifier_name == "Bagging":
        clf = BaggingClassifier(
            n_estimators=trial.suggest_int("bagging_n_estimators", 10, 200),
            max_samples=trial.suggest_float("bagging_max_samples", 0.5, 1.0),
            max_features=trial.suggest_float("bagging_max_features", 0.5, 1.0),
            random_state=42
        )

    elif classifier_name == "AdaBoost":
        clf = AdaBoostClassifier(
            n_estimators=trial.suggest_int("adaboost_n_estimators", 50, 300),
            learning_rate=trial.suggest_float("adaboost_learning_rate", 0.01, 2.0, log=True),
            random_state=42
        )

    elif classifier_name == "GradientBoosting":
        clf = GradientBoostingClassifier(
            n_estimators=trial.suggest_int("gb_n_estimators", 50, 300),
            learning_rate=trial.suggest_float("gb_learning_rate", 0.01, 0.5, log=True),
            max_depth=trial.suggest_int("gb_max_depth", 2, 6),
            subsample=trial.suggest_float("gb_subsample", 0.5, 1.0),
            random_state=42
        )

    elif classifier_name == "XGBoost":
        clf = xgb.XGBClassifier(
            n_estimators=trial.suggest_int("xgb_n_estimators", 50, 300),
            learning_rate=trial.suggest_float("xgb_learning_rate", 0.01, 0.5, log=True),
            max_depth=trial.suggest_int("xgb_max_depth", 2, 8),
            subsample=trial.suggest_float("xgb_subsample", 0.5, 1.0),
            random_state=42,
            # eval_metric avoids a deprecation warning in multiclass problems.
            # verbosity=0 suppresses XGBoost's internal logs during tuning.
            eval_metric='mlogloss',
            verbosity=0
        )

    elif classifier_name == "RandomForest":
        clf = RandomForestClassifier(
            n_estimators=trial.suggest_int("rf_n_estimators", 50, 300),
            max_depth=trial.suggest_int("rf_max_depth", 2, 20),
            max_features=trial.suggest_categorical("rf_max_features", ["sqrt", "log2"]),
            min_samples_split=trial.suggest_int("rf_min_samples_split", 2, 10),
            random_state=42
        )

    else:  # Stacking
        clf = StackingClassifier(
            estimators=[
                ('lr', LogisticRegression(max_iter=1000, random_state=42)),
                ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
                ('svc', SVC(probability=True, random_state=42))
            ],
            final_estimator=LogisticRegression(
                C=trial.suggest_float("stacking_C", 1e-3, 10.0, log=True),
                max_iter=1000,
                random_state=42
            ),
            cv=3
        )

    # We use y_train_encoded instead of y_train so that XGBoost receives
    # integer labels. All scikit-learn models are equally happy with integers.
    score = cross_val_score(
        clf, X_train_processed, y_train_encoded, cv=3, scoring='accuracy', n_jobs=-1
    )
    return score.mean()


# ── RUN STUDY ─────────────────────────────────────────────────────────────────

optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    direction="maximize",
    pruner=MedianPruner()
)
study.optimize(objective, n_trials=100)


# ── RESULTS PER MODEL ─────────────────────────────────────────────────────────

print("=" * 60)
print("BEST CV ACCURACY PER MODEL")
print("=" * 60)

model_names = [
    "VotingHard", "VotingSoft", "Bagging", "AdaBoost",
    "GradientBoosting", "XGBoost", "RandomForest", "Stacking"
]

for name in model_names:
    model_trials = [
        t for t in study.trials
        if t.params.get("classifier") == name and t.value is not None
    ]
    if model_trials:
        best_trial = max(model_trials, key=lambda t: t.value)
        print(f"{name:20s} → Best CV Accuracy: {best_trial.value:.4f}  ({len(model_trials)} trials)")

# For each model, print the best hyperparameters found by Optuna.
# We filter out the 'classifier' key since it's not a hyperparameter itself.
print()
print("=" * 60)
print("BEST HYPERPARAMETERS PER MODEL")
print("=" * 60)

for name in model_names:
    model_trials = [
        t for t in study.trials
        if t.params.get("classifier") == name and t.value is not None
    ]
    if model_trials:
        best_trial = max(model_trials, key=lambda t: t.value)
        model_params = {k: v for k, v in best_trial.params.items() if k != "classifier"}
        print(f"\n{name}:")
        for param, value in model_params.items():
            print(f"  {param}: {value}")

print()
print(f"Overall best model: {study.best_params['classifier']}")
print(f"Overall best CV accuracy: {study.best_value:.4f}")


# ── HELPER: build a model from its best trial params ─────────────────────────

def build_model(classifier_name, params):
    """Reconstruct a model from its best Optuna params."""

    if classifier_name == "VotingHard":
        return VotingClassifier(
            estimators=[
                ('lr', LogisticRegression(max_iter=1000, random_state=42)),
                ('rf', RandomForestClassifier(n_estimators=params["voting_hard_rf_n_estimators"], random_state=42)),
                ('svc', SVC(random_state=42))
            ],
            voting='hard'
        )
    elif classifier_name == "VotingSoft":
        return VotingClassifier(
            estimators=[
                ('lr', LogisticRegression(max_iter=1000, random_state=42)),
                ('rf', RandomForestClassifier(n_estimators=params["voting_soft_rf_n_estimators"], random_state=42)),
                ('svc', SVC(probability=True, random_state=42))
            ],
            voting='soft'
        )
    elif classifier_name == "Bagging":
        return BaggingClassifier(
            n_estimators=params["bagging_n_estimators"],
            max_samples=params["bagging_max_samples"],
            max_features=params["bagging_max_features"],
            random_state=42
        )
    elif classifier_name == "AdaBoost":
        return AdaBoostClassifier(
            n_estimators=params["adaboost_n_estimators"],
            learning_rate=params["adaboost_learning_rate"],
            random_state=42
        )
    elif classifier_name == "GradientBoosting":
        return GradientBoostingClassifier(
            n_estimators=params["gb_n_estimators"],
            learning_rate=params["gb_learning_rate"],
            max_depth=params["gb_max_depth"],
            subsample=params["gb_subsample"],
            random_state=42
        )
    elif classifier_name == "XGBoost":
        return xgb.XGBClassifier(
            n_estimators=params["xgb_n_estimators"],
            learning_rate=params["xgb_learning_rate"],
            max_depth=params["xgb_max_depth"],
            subsample=params["xgb_subsample"],
            random_state=42,
            eval_metric='mlogloss',
            verbosity=0
        )
    elif classifier_name == "RandomForest":
        return RandomForestClassifier(
            n_estimators=params["rf_n_estimators"],
            max_depth=params["rf_max_depth"],
            max_features=params["rf_max_features"],
            min_samples_split=params["rf_min_samples_split"],
            random_state=42
        )
    else:  # Stacking
        return StackingClassifier(
            estimators=[
                ('lr', LogisticRegression(max_iter=1000, random_state=42)),
                ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
                ('svc', SVC(probability=True, random_state=42))
            ],
            final_estimator=LogisticRegression(
                C=params["stacking_C"],
                max_iter=1000,
                random_state=42
            ),
            cv=3
        )


# ── FULL EVALUATION OF ALL MODELS ON TEST SET ────────────────────────────────
# We retrain every model with its best hyperparameters on the full train set
# and evaluate on the test set. This gives a fair comparison across all models
# using the same metrics: Accuracy, Precision, Recall, F1 and MCC.
# We use macro averaging for multiclass — it treats all classes equally,
# which is informative especially in the presence of class imbalance.

print()
print("=" * 60)
print("FULL EVALUATION ON TEST SET — ALL MODELS")
print("=" * 60)

comparison_rows = []

for name in model_names:
    model_trials = [
        t for t in study.trials
        if t.params.get("classifier") == name and t.value is not None
    ]
    if not model_trials:
        print(f"{name:20s} → no successful trials, skipping.")
        continue

    best_trial = max(model_trials, key=lambda t: t.value)
    model = build_model(name, best_trial.params)

    # Fit on full train set with encoded labels
    model.fit(X_train_processed, y_train_encoded)
    y_pred = model.predict(X_test_processed)

    acc  = accuracy_score(y_test_encoded, y_pred)
    prec = precision_score(y_test_encoded, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_test_encoded, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_test_encoded, y_pred, average='macro', zero_division=0)
    mcc  = matthews_corrcoef(y_test_encoded, y_pred)

    comparison_rows.append({
        "Model":      name,
        "Accuracy":   round(acc,  4),
        "Precision":  round(prec, 4),
        "Recall":     round(rec,  4),
        "F1 (macro)": round(f1,   4),
        "MCC":        round(mcc,  4),
    })

    print(f"\n{'─'*40}")
    print(f"Model: {name}")
    print(f"  Accuracy:   {acc:.4f}")
    print(f"  Precision:  {prec:.4f}  (macro)")
    print(f"  Recall:     {rec:.4f}  (macro)")
    print(f"  F1 Score:   {f1:.4f}  (macro)")
    print(f"  MCC:        {mcc:.4f}")
    print()
    # Per-class breakdown — target_names restores original string labels
    print(classification_report(y_test_encoded, y_pred, target_names=le.classes_))

# ── COMPARISON TABLE ──────────────────────────────────────────────────────────
# Summary table sorted by F1 score so it's easy to spot the best model overall.

print()
print("=" * 60)
print("COMPARISON TABLE (sorted by F1 macro)")
print("=" * 60)

df_results = pd.DataFrame(comparison_rows).sort_values("F1 (macro)", ascending=False)
df_results = df_results.reset_index(drop=True)
df_results.index += 1  # Start ranking from 1
print(df_results.to_string())

print()
best_row = df_results.iloc[0]
print(f"Best model overall: {best_row['Model']}  —  F1: {best_row['F1 (macro)']:.4f}  |  Accuracy: {best_row['Accuracy']:.4f}  |  MCC: {best_row['MCC']:.4f}")

BEST RESULTS PER MODEL
VotingHard           → Best CV Accuracy: 0.3965  (6 trials)
VotingSoft           → Best CV Accuracy: 0.3547  (7 trials)
Bagging              → Best CV Accuracy: 0.3816  (6 trials)
AdaBoost             → Best CV Accuracy: 0.3965  (7 trials)
GradientBoosting     → Best CV Accuracy: 0.3966  (6 trials)
XGBoost              → Best CV Accuracy: 0.3966  (9 trials)
RandomForest         → Best CV Accuracy: 0.3966  (7 trials)
Stacking             → Best CV Accuracy: 0.3966  (52 trials)

Overall best model: Stacking
Overall best CV accuracy: 0.3966
Best params: {'classifier': 'Stacking', 'stacking_C': 3.494470519107966}

FINAL EVALUATION ON TEST SET
Model: Stacking
Test Accuracy: 0.3963

Classification Report:
               precision    recall  f1-score   support

         Free       0.00      0.00      0.00       393
     Freemium       0.40      1.00      0.57       766
  Open Source       0.00      0.00      0.00       191
Pay-as-you-go       0.00      0.00      0.00   